In [ ]:
import os
GEMINI_API_KEY = os.environ["GEMINI_API_KEY"]

In [ ]:
import os
import pandas as pd
from google import genai
from google.genai import types
from PIL import Image
from tqdm import tqdm
import json
import time

# =====================================================================
# 1. Gemini API 클라이언트 설정 (새로운 SDK 방식)
# =====================================================================
import os
GEMINI_API_KEY = os.environ["GEMINI_API_KEY"]

# 새로운 Client 객체 생성
client = genai.Client(api_key=GEMINI_API_KEY)

# 최신 Flash 모델명 사용 (버전에 따라 gemini-2.5-flash 등이 권장됨)
MODEL_ID = 'gemini-2.5-flash-lite' 

# =====================================================================
# 2. 파일 경로 설정
# =====================================================================
INPUT_CSV_PATH = 'runtime/coco_sharegpt_integrated.csv'
OUTPUT_CSV_PATH = 'runtime/prototype_filtered_images.csv'

# 테스트할 이미지 파일 리스트
TEST_FILES = [
    "000000000078.jpg", "000000000081.jpg", "000000000071.jpg", 
    "000000000109.jpg", "000000000154.jpg", "000000000201.jpg", 
    "000000000322.jpg", "000000000328.jpg", "000000000309.jpg", 
    "000000000446.jpg", "000000000490.jpg", '000000000370.jpg','000000002624.jpg','000000021963.jpg','000000020541.jpg','000000022086.jpg','000000208931.jpg'
]

# =====================================================================
# 3. 프롬프트 정의
# =====================================================================
SYSTEM_PROMPT = """
**Role:** You are an expert data annotator specializing in visual temporal grounding. Your task is to analyze an image and its accompanying description to determine if there are **definitive, unambiguous visual clues** indicating the time of day (Day vs. Night) or the season.

**Task:** Output a JSON object classifying whether the image contains reliable temporal clues.

**[Strict Evaluation Criteria]**

**Classify as `1` (True - Temporal Clues Present) ONLY IF at least one of the following is clearly visible AND the image is in full color:**
1. **Daytime (General Outdoors):**
   - Any broad outdoor scene clearly illuminated by general daylight. This includes overcast/cloudy days where the landscape is fully visible. Distinct shadows or blue skies are NOT strictly required if the overall scene is obviously daytime.
2. **Specific Time of Day (Natural Light):**
   - Golden hour phenomena (e.g., visible sunrise, sunset, deep orange/pink skies).
   - Nighttime indicators in an outdoor setting (e.g., pitch-black sky, visible stars, streetlights/car headlights illuminating a dark environment).
   - Indoor scenes ONLY IF an outside window clearly reveals the exterior environment (e.g., broad daylight, sunset, or night sky outside).
3. **Season (Nature & Weather):**
   - Unambiguous seasonal weather phenomena (e.g., accumulated snow on the ground, blooming cherry blossoms, fully turning autumn foliage).
4. **Season (Apparel - Outdoors ONLY):**
   - People outdoors wearing extreme weather clothing (e.g., heavy winter puffer jackets) or highly specific summer attire (e.g., swimwear at a beach).

**Classify as `0` (False - No Reliable Temporal Clues) IF the image falls into any of these categories:**
1. **Black & White / Grayscale Images:** The image lacks color information. Color is essential for consistently judging natural light, golden hour phenomena, and seasonal changes. If it's not a full-color image, output 0.
2. **Ambiguous Indoor Lighting & Shadows:** Indoor scenes without a clear view of the outside world through a window. WARNING: Beams of light, harsh shadows on walls, or bright illumination indoors are NOT reliable clues, as they can be created by studio lights, flash photography, or strong artificial night lights.
3. **Timeless Subjects:** Close-up macro shots of objects, animals, food, text, screens, or studio portraits where the broader surrounding environment is cropped out or lacks lighting context.
4. **Ambiguous Apparel:** People wearing generic clothing (e.g., t-shirts, hoodies) indoors, or in environments where the outside weather cannot be corroborated.
5. **Completely Featureless Skies (No Context):** A tight shot of purely grey clouds or blank sky with absolutely no ground/landscape visible to indicate if it's daytime.
6. **Clocks/Watches:** A visible clock face without environmental corroboration.

**[Output Format]**
You must return ONLY a valid JSON object. Do not include markdown formatting like ```json or any conversational text.
{
  "is_temporal": <integer 1 or 0>,
  "reason": "<A brief, 1-2 sentence explanation in English detailing exactly which visual/textual element triggered the 1 or 0>"
}
"""

def analyze_image(image_path, description):
    try:
        img = Image.open(image_path).convert('RGB')
        prompt = f"{SYSTEM_PROMPT}\n\n[image description]\n{description}\n\nPlease analyze the image and description to determine if there are reliable temporal clues."
        
        for attempt in range(3):
            try:
                # 새로운 SDK의 API 호출 방식
                response = client.models.generate_content(
                    model=MODEL_ID,
                    contents=[img, prompt],
                    config=types.GenerateContentConfig(
                        response_mime_type="application/json",
                        temperature=0.1
                    )
                )
                
                result = json.loads(response.text)
                is_temporal_val = result.get('is_temporal', 0)
                return bool(is_temporal_val), result.get('reason', '')
                
            except Exception as e:
                if attempt == 2:
                    print(f"Error calling Gemini API for {image_path}: {e}")
                    return False, "API Error"
                time.sleep(2) 
                
    except Exception as e:
        print(f"Error loading image {image_path}: {e}")
        return False, "Image Load Error"

def main():
    print(f"데이터 로딩 중: {INPUT_CSV_PATH}")
    df = pd.read_csv(INPUT_CSV_PATH)
    
    # 테스트 파일 리스트에 있는 데이터만 추출
    prototype_df = df[df['File Name'].isin(TEST_FILES)].copy()
    
    if prototype_df.empty:
        print("경고: 테스트 파일 리스트에 해당하는 이미지가 CSV 파일에 없습니다.")
        return

    results = []
    print(f"프로토타입 테스트: 총 {len(prototype_df)}개의 이미지 분석 시작...")
    
    for index, row in tqdm(prototype_df.iterrows(), total=len(prototype_df)):
        file_name = row['File Name']
        description = row['Description']
        full_path = row['Full Path']
        
        if pd.isna(full_path) or not os.path.exists(full_path):
            has_clue, reason = False, "File Not Found"
        else:
            has_clue, reason = analyze_image(full_path, description)
            
        results.append({
            'File Name': file_name,
            'Description': description,
            'DateTime': row.get('DateTime', ''),
            'Location': row.get('Location', ''),
            'Full Path': full_path,
            'Status': row.get('Status', ''),
            'Is_Temporal': has_clue,
            'Reason': reason
        })
        
        time.sleep(1)

    # 최종 결과 저장
    output_df = pd.DataFrame(results)
    output_df.to_csv(OUTPUT_CSV_PATH, index=False, encoding='utf-8-sig')
    
    print(f"\n프로토타입 분석 완료! 결과가 저장되었습니다: {OUTPUT_CSV_PATH}")
    
    # 터미널에서 결과 바로 확인
    print("\n[테스트 결과 요약]")
    for res in results:
        status = "✅ True" if res['Is_Temporal'] else "❌ False"
        print(f"- {res['File Name']}: {status} | {res['Reason']}")

if __name__ == "__main__":
    main()

In [ ]:
import pandas as pd
import os

# 파일 경로 설정
FILE_PATH = 'runtime/full_filtered_images.csv'

def cleanup_error_rows():
    if not os.path.exists(FILE_PATH):
        print("❌ 파일을 찾을 수 없습니다. 경로를 확인해주세요.")
        return

    # 1. CSV 로드
    df = pd.read_csv(FILE_PATH)
    original_count = len(df)
    
    # 2. 'Reason' 컬럼에 'API Error' 또는 '404'가 포함된 행 필터링
    # 에러가 없는 정상적인 행만 남깁니다.
    # (Reason이 비어있는 경우를 대비해 fillna('') 처리)
    clean_df = df[~df['Reason'].fillna('').str.contains('API Error|404|429|NOT_FOUND|Fatal Error', case=False)]
    
    deleted_count = original_count - len(clean_df)
    
    # 3. 결과 저장
    if deleted_count > 0:
        clean_df.to_csv(FILE_PATH, index=False, encoding='utf-8-sig')
        print(f"✅ 정제 완료!")
        print(f"   - 원래 데이터: {original_count}개")
        print(f"   - 삭제된 에러 데이터: {deleted_count}개")
        print(f"   - 남은 정상 데이터: {len(clean_df)}개")
    else:
        print("🔍 삭제할 에러 데이터가 발견되지 않았습니다.")

if __name__ == "__main__":
    cleanup_error_rows()

In [ ]:
import pandas as pd
import numpy as np
import os

# 파일 경로
csv_path = 'runtime/full_filtered_images.csv'

def is_empty(val):
    """값이 정말로 비어있는지 체크 (NaN, 공백, 'nan' 문자열 등)"""
    if pd.isna(val): return True
    s_val = str(val).strip().lower()
    return s_val in ["", "nan", "none", "0", "null"]

try:
    df = pd.read_csv(csv_path)

    # 1. Status 계산 함수 (사용자 정의 기준 적용)
    def assign_new_status(row):
        has_dt = not is_empty(row.get('DateTime'))
        has_loc = not is_empty(row.get('Location'))
        
        if has_dt and has_loc:   return 3  # 둘 다 있음
        elif has_dt:             return 1  # 시간만 있음
        elif has_loc:            return 2  # 위치만 있음
        else:                    return 0  # 둘 다 없음

    # 2. Status 값 덮어씌우기
    df['Status'] = df.apply(assign_new_status, axis=1)


    # 4. 모든 Status(0,1,2,3) 분포 강제 확인
    print("\n📊 [최종 Status 분포 리포트]")
    print("-" * 40)
    
    # 0, 1, 2, 3이 다 나오도록 카운트
    status_counts = df['Status'].value_counts().reindex([0, 1, 2, 3], fill_value=0)
    status_names = {3: "시간+위치", 1: "시간만", 2: "위치만", 0: "정보없음"}

    for s_val, count in status_counts.items():
        ratio = (count / len(df)) * 100
        print(f"📍 Status {s_val} ({status_names[s_val]}): {count}개 ({ratio:.1f}%)")

    # 5. Is_Temporal과의 교차 분석 (상세 요약)
    summary = pd.crosstab(df['Status'], df['Is_Temporal'], dropna=False).reindex([0,1,2,3], fill_value=0)
    print("\n💡 Is_Temporal 합산 결과:")
    print(summary)

except Exception as e:
    print(f"⚠️ 오류 발생: {e}")

In [ ]:
import pandas as pd

# 1. 데이터셋 불러오기
# 파일 경로가 다를 경우 실제 경로로 수정하여 사용하세요.
file1_path = 'runtime/coco_sharegpt_integrated_yfcc.csv'
file2_path = 'runtime/full_filtered_images.csv'

df2 = pd.read_csv(file1_path)
df1 = pd.read_csv(file2_path)

# 2. 'File Name' 컬럼을 기준으로 병합 (Inner Join)
# 공통된 File Name이 있는 경우에만 합쳐집니다.
# 중복되는 컬럼명(DateTime, Status 등)은 df1의 것을 유지합니다.
merged_df = pd.merge(df1, df2, on='File Name', suffixes=('', '_remove'))

# 3. 요청하신 최종 컬럼만 선택
final_columns = [
    'File Name', 
    'Description', 
    'DateTime', 
    'Full Path', 
    'Status', 
    'latitude', 
    'longitude', 
    'Is_Temporal'
]

# 데이터프레임 필터링
final_df = merged_df[final_columns]

# 4. 최종 결과 저장
final_df.to_csv('final_data.csv', index=False, encoding='utf-8-sig')

print("병합이 완료되었습니다. 'final_data.csv' 파일이 생성되었습니다.")

In [ ]:
final_df.groupby('Status')['Is_Temporal'].value_counts()

In [ ]:
import os
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
from datetime import datetime, timedelta
import math
import numpy as np
from tqdm import tqdm
import calendar

# --- [0. 설정] ---
RESULT_SAVE_DIR = "analysis_result"
MODEL_SAVE_DIR = "saved_models"
os.makedirs(RESULT_SAVE_DIR, exist_ok=True)
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

# --- [1. 시간 변환 및 오차 계산 함수] ---
DAY_TO_MONTH = 1 / (365 / 12)
MINUTE_TO_HOUR = 1 / 60

def angular_time_representation(T):
    month, day, hour, minute = torch.chunk(T.float(), 4, dim=1)
    month_d = (month - 1) + (day - 1) * DAY_TO_MONTH
    hour_d = hour + minute * MINUTE_TO_HOUR 
    theta = 2 * math.pi * month_d / 12 - math.pi
    phi = 2 * math.pi * hour_d / 24 - math.pi
    return torch.cat((theta, phi), dim=1)

def inverse_angular_time(theta, phi):
    month_d = ((theta + math.pi) / (2 * math.pi)) * 12
    month = max(1, min(12, int(month_d) + 1))
    day_fraction = (month_d - int(month_d)) / DAY_TO_MONTH
    last_day = calendar.monthrange(2026, month)[1]
    day = max(1, min(last_day, int(day_fraction) + 1))
    hour_d = ((phi + math.pi) / (2 * math.pi)) * 24
    hour = int(hour_d) % 24
    minute = int((hour_d - int(hour_d)) * 60) % 60
    return datetime(2026, month, day, hour, minute, 0)

def calculate_time_diff(actual_str, pred_dt):
    """실제 문자열 날짜와 예측 datetime 객체 사이의 오차(초) 계산"""
    actual_dt = pd.to_datetime(actual_str)
    try:
        actual_2026 = actual_dt.replace(year=2026, second=0, microsecond=0)
    except ValueError: # 윤달(2월 29일) 대응
        actual_2026 = actual_dt.replace(year=2026, month=2, day=28, second=0, microsecond=0)
    
    diff = abs((actual_2026 - pred_dt).total_seconds())
    if diff > 15768000: diff = 31536000 - diff # 1년 주기 순환 보정
    return diff

def cyclic_loss(pred, target):
    return (1 - torch.cos(pred[:, 0] - target[:, 0]) + 1 - torch.cos(pred[:, 1] - target[:, 1])).mean()

# --- [2. 데이터셋 및 모델 정의] ---
class FullSequenceDataset(Dataset):
    def __init__(self, pt_dir, csv_path, layer_idx):
        self.pt_dir = pt_dir
        self.layer_key = f"hidden_states{layer_idx}"
        df = pd.read_csv(csv_path)
        df = df[df['Status'].astype(str) != '0'].copy()
        self.data_list = []
        for _, row in df.iterrows():
            img_name = str(row['File Name']).strip()
            pt_path = os.path.join(pt_dir, f"{img_name}.pt")
            if os.path.exists(pt_path):
                try:
                    dt = pd.to_datetime(str(row['DateTime']).strip())
                    self.data_list.append({'pt_path': pt_path, 'time_vec': [dt.month, dt.day, dt.hour, dt.minute], 'row_data': row.to_dict()})
                except: continue
    def __len__(self): return len(self.data_list)
    def __getitem__(self, idx):
        item = self.data_list[idx]
        data = torch.load(item['pt_path'], map_location='cpu')
        return data[self.layer_key].to(torch.float32), torch.tensor(item['time_vec'], dtype=torch.float32), item['row_data']

def collate_fn(batch):
    embeddings, times, rows = zip(*batch)
    padded = pad_sequence(embeddings, batch_first=True)
    mask = torch.arange(padded.size(1))[None, :] < torch.tensor([e.size(0) for e in embeddings])[:, None]
    return padded, torch.stack(times), mask, rows

class SequenceProbeModel(nn.Module):
    def __init__(self, input_dim=4096):
        super().__init__()
        self.attention = nn.Sequential(nn.Linear(input_dim, 512), nn.Tanh(), nn.Linear(512, 1))
        self.head = nn.Sequential(nn.Linear(input_dim, 1024), nn.ReLU(), nn.Dropout(0.2), nn.Linear(1024, 2))
    def forward(self, x, mask):
        attn_scores = self.attention(x).squeeze(-1).masked_fill(~mask, -1e9)
        attn_weights = torch.softmax(attn_scores, dim=1).unsqueeze(-1)
        return self.head(torch.sum(attn_weights * x, dim=1))

# --- [3. 메인 분석 함수] ---
def run_integrated_analysis(pt_dir, csv_path):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    layers = [9, 19, 24]
    weights = {9: 0.34, 19: 0.33, 24: 0.33}
    ensemble_data = {}

    for layer_idx in layers:
        print(f"\n{'='*20} 🟢 Layer {layer_idx} 학습 및 검증 시작 {'='*20}")
        dataset = FullSequenceDataset(pt_dir, csv_path, layer_idx)
        train_len = int(0.8 * len(dataset))
        val_len = len(dataset) - train_len
        train_ds, val_ds = random_split(dataset, [train_len, val_len], generator=torch.Generator().manual_seed(42))
        
        train_loader = DataLoader(train_ds, batch_size=8, shuffle=True, collate_fn=collate_fn)
        val_loader = DataLoader(val_ds, batch_size=1, collate_fn=collate_fn)
        
        model = SequenceProbeModel().to(device)
        optimizer = optim.Adam(model.parameters(), lr=1e-4)
        model_path = os.path.join(MODEL_SAVE_DIR, f"best_layer_{layer_idx}.pth")
        
        best_val_error = float('inf')

        for epoch in range(10):
            # [Train Phase]
            model.train()
            train_loss = 0
            for h, t, m, _ in tqdm(train_loader, desc=f"L{layer_idx} Epoch {epoch+1} [Train]", leave=False):
                h, t, m = h.to(device), t.to(device), m.to(device)
                optimizer.zero_grad()
                loss = cyclic_loss(model(h, m), angular_time_representation(t))
                loss.backward()
                optimizer.step()
                train_loss += loss.item()

            # [Validation Phase]
            model.eval()
            val_loss = 0
            total_error_sec = 0
            with torch.no_grad():
                for h, t, m, rows in val_loader:
                    h, t, m = h.to(device), t.to(device), m.to(device)
                    pred = model(h, m)
                    val_loss += cyclic_loss(pred, angular_time_representation(t)).item()
                    
                    # 실제 시간 오차(초) 계산
                    pred_np = pred.cpu().numpy()[0]
                    p_dt = inverse_angular_time(pred_np[0], pred_np[1])
                    total_error_sec += calculate_time_diff(rows[0]['DateTime'], p_dt)

            avg_train_loss = train_loss / len(train_loader)
            avg_val_loss = val_loss / len(val_loader)
            avg_val_error_sec = total_error_sec / len(val_loader)
            
            # 오차 포맷팅 (HH:MM:SS)
            error_msg = str(timedelta(seconds=int(avg_val_error_sec)))

            print(f"Epoch {epoch+1:2d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | ⏱️ 평균 오차: {error_msg}")

            # 최적 모델 저장
            if avg_val_error_sec < best_val_error:
                best_val_error = avg_val_error_sec
                torch.save(model.state_dict(), model_path)
                print(f"   ✨ Best Model 저장됨! (오차: {error_msg})")

        # [최종 추론 및 리포트 생성]
        print(f"🔍 Layer {layer_idx} 최종 리포트 생성 중...")
        model.load_state_dict(torch.load(model_path, map_location=device))
        model.eval()
        
        layer_results = []
        with torch.no_grad():
            for h, t, m, rows in tqdm(val_loader, desc=f"L{layer_idx} Final Inference", leave=False):
                h, m = h.to(device), m.to(device)
                pred = model(h, m).cpu().numpy()[0]
                row = rows[0]
                f_name = row['File Name']
                
                if f_name not in ensemble_data: ensemble_data[f_name] = {'theta': 0, 'phi': 0, 'row': row}
                ensemble_data[f_name]['theta'] += pred[0] * weights[layer_idx]
                ensemble_data[f_name]['phi'] += pred[1] * weights[layer_idx]
                
                p_dt = inverse_angular_time(pred[0], pred[1])
                layer_results.append({
                    'File Name': f_name, 
                    'Actual': row['DateTime'], 
                    'Predicted': p_dt.strftime('%m-%d %H:%M'),
                    'Error': str(timedelta(seconds=int(calculate_time_diff(row['DateTime'], p_dt))))
                })
        pd.DataFrame(layer_results).to_csv(os.path.join(RESULT_SAVE_DIR, f"individual_layer_{layer_idx}.csv"), index=False, encoding='utf-8-sig')

    # [앙상블 리포트 생성]
    print(f"\n🏆 최종 앙상블 리포트 생성 중...")
    ens_results = []
    for f_name, data in ensemble_data.items():
        p_dt = inverse_angular_time(data['theta'], data['phi'])
        diff_sec = calculate_time_diff(data['row']['DateTime'], p_dt)
        res = data['row'].copy()
        res.update({
            'Actual_Cycle': pd.to_datetime(data['row']['DateTime']).strftime('%m-%d %H:%M'),
            'Predicted_Cycle': p_dt.strftime('%m-%d %H:%M'),
            'Error_Format': str(timedelta(seconds=int(diff_sec)))
        })
        ens_results.append(res)

    pd.DataFrame(ens_results).to_csv(os.path.join(RESULT_SAVE_DIR, "final_ensemble_report.csv"), index=False, encoding='utf-8-sig')
    print(f"✨ 완료! 모든 결과는 {RESULT_SAVE_DIR}에 저장되었습니다.")

if __name__ == "__main__":
    PT_DIR = 'runtime/embedding_results_448_pt_coco'
    CSV_PATH = 'runtime/final_data.csv'
    run_integrated_analysis(PT_DIR, CSV_PATH)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import os

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

def perform_detailed_eda(csv_path):
    if not os.path.exists(csv_path):
        print(f"❌ 파일을 찾을 수 없습니다: {csv_path}")
        return

    # 1. 데이터 로드
    df = pd.read_csv(csv_path)
    
    # Error 문자열을 초 단위 숫자로 변환
    df['Error_Total_Seconds'] = pd.to_timedelta(df['Error']).dt.total_seconds()

    # 날짜 파싱 로직
    def parse_actual(x):
        try:
            return pd.to_datetime(x)
        except:
            return None

    def parse_predicted(x):
        try:
            return datetime.strptime(f"2026-{x}", "%Y-%m-%d %H:%M")
        except:
            return None

    df['Actual_DT'] = df['Actual'].apply(parse_actual)
    df['Pred_DT'] = df['Predicted'].apply(parse_predicted)
    
    # 결측치 제거
    df = df.dropna(subset=['Actual_DT', 'Pred_DT'])
    
    # 분석용 속성 추출
    df['Actual_Month'] = df['Actual_DT'].dt.month
    df['Actual_Hour'] = df['Actual_DT'].dt.hour
    df['Error_Hours'] = df['Error_Total_Seconds'] / 3600
    df['Error_Days'] = df['Error_Hours'] / 24

    print("\n=== 📋 레이어별 예측 오차 요약 통계 ===")
    stats = df['Error_Total_Seconds'].describe()
    print(f"평균 오차: {timedelta(seconds=int(stats['mean']))}")
    print(f"중앙 오차: {timedelta(seconds=int(stats['50%']))}")
    print("="*30)

    # --- 시각화 시작 ---
    fig = plt.figure(figsize=(18, 12))
    
    # (1) 오차 분포
    plt.subplot(2, 2, 1)
    sns.histplot(df['Error_Days'], bins=30, kde=True, color='skyblue')
    plt.title('오차 분포 (단위: 일)')

    # (2) 실제 날짜 vs 예측 날짜 산점도 (윤달 예외 처리 적용)
    plt.subplot(2, 2, 2)
    
    # 🛠️ 수정 포인트: 2월 29일인 경우 2월 28일로 바꿔서 2026년 날짜를 생성합니다.
    def to_2026(dt):
        try:
            return dt.replace(year=2026)
        except ValueError: # 2월 29일 에러 발생 시
            return dt.replace(year=2026, month=2, day=28)

    df['Actual_2026'] = df['Actual_DT'].apply(to_2026)
    df['Actual_DayOfYear'] = df['Actual_2026'].dt.dayofyear
    df['Pred_DayOfYear'] = df['Pred_DT'].dt.dayofyear
    
    plt.scatter(df['Actual_DayOfYear'], df['Pred_DayOfYear'], alpha=0.5, c=df['Error_Days'], cmap='viridis')
    plt.plot([0, 366], [0, 366], 'r--', alpha=0.7)
    plt.title('실제 날짜 vs 예측 날짜 (주기 비교)')
    plt.xlabel('실제 날짜 (Day of Year)')
    plt.ylabel('예측 날짜 (Day of Year)')
    plt.colorbar(label='오차 (Days)')

    # (3) 월별 평균 오차
    plt.subplot(2, 2, 3)
    sns.barplot(data=df, x='Actual_Month', y='Error_Days', palette='coolwarm')
    plt.title('촬영 월별 평균 예측 오차')

    # (4) 시간대별 평균 오차
    plt.subplot(2, 2, 4)
    sns.lineplot(data=df, x='Actual_Hour', y='Error_Hours', marker='o', color='orange')
    plt.xticks(range(0, 24))
    plt.grid(True, alpha=0.3)
    plt.title('촬영 시간대별 평균 예측 오차')

    plt.tight_layout()
    plt.savefig('layer_analysis_plots_fixed.png')
    plt.show()

    # 오차 샘플 출력
    print("\n=== 🚀 가장 잘 맞춘 예측 (Top 5) ===")
    print(df.sort_values('Error_Total_Seconds').head(5)[['File Name', 'Actual', 'Predicted', 'Error']])
    print("\n=== 🚀 가장 못 맞춘 예측 (Top 5) ===")
    print(df.sort_values('Error_Total_Seconds', ascending=False).head(5)[['File Name', 'Actual', 'Predicted', 'Error']])
if __name__ == "__main__":
    path = 'runtime/individual_layer_9.csv'
    perform_detailed_eda(path)

In [ ]:
import os
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from datetime import datetime, timedelta
import math
import numpy as np
from tqdm import tqdm
import calendar

# --- [0. 경로 및 설정] ---
# 학습 때 사용했던 가중치 파일 경로
MODEL_PATH = r"saved_models\best_layer_9.pth" 
# 추론할 새로운 데이터셋 경로
NEW_CSV_PATH = 'runtime/results.csv'
# 임베딩(.pt) 파일이 들어있는 폴더
PT_DIR = 'runtime/embedding_results_448_pt'
# 결과 저장 경로
OUTPUT_SAVE_PATH = r"analysis_result\inference_results_layer9_new.csv"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- [1. 시간 변환 함수 (초 단위 제외)] ---
DAY_TO_MONTH = 1 / (365 / 12)
MINUTE_TO_HOUR = 1 / 60

def inverse_angular_time(theta, phi):
    """예측된 각도를 다시 시간으로 (2026년 기준)"""
    month_d = ((theta + math.pi) / (2 * math.pi)) * 12
    month = max(1, min(12, int(month_d) + 1))
    day_fraction = (month_d - int(month_d)) / DAY_TO_MONTH
    last_day = calendar.monthrange(2026, month)[1]
    day = max(1, min(last_day, int(day_fraction) + 1))
    hour_d = ((phi + math.pi) / (2 * math.pi)) * 24
    hour = int(hour_d) % 24
    minute = int((hour_d - int(hour_d)) * 60) % 60
    return datetime(2026, month, day, hour, minute, 0)

def calculate_time_diff(actual_dt, pred_dt):
    """오차 계산 (윤달 대응 및 순환 보정)"""
    try:
        actual_2026 = actual_dt.replace(year=2026, second=0, microsecond=0)
    except ValueError:
        actual_2026 = actual_dt.replace(year=2026, month=2, day=28, second=0, microsecond=0)
    
    diff = abs((actual_2026 - pred_dt).total_seconds())
    if diff > 15768000: diff = 31536000 - diff
    return diff

# --- [2. 추론용 데이터셋 정의] ---
class InferenceDataset(Dataset):
    def __init__(self, pt_dir, csv_path, layer_idx=9):
        self.pt_dir = pt_dir
        self.layer_key = f"hidden_states{layer_idx}"
        
        # CSV 로드 및 Status 필터링
        df = pd.read_csv(csv_path)
        df = df[df['Status'].astype(str) != '0'].copy()
        
        self.data_list = []
        print(f"🔍 데이터 로딩 및 날짜 형식 변환 중...")
        
        for _, row in tqdm(df.iterrows(), total=len(df), desc="Parsing CSV"):
            # 파일명 추출 (Full Path에서 추출하거나 File Name 컬럼 사용)
            img_name = str(row['File Name']).strip()
            if not img_name or img_name == 'nan':
                img_name = os.path.basename(row['Full Path'])
            
            pt_path = os.path.join(pt_dir, f"{img_name}.pt")
            
            if os.path.exists(pt_path):
                try:
                    # 🛠️ 시간 형식 처리 (YYYY:MM:DD HH:MM:SS 대응)
                    dt_str = str(row['DateTime']).strip()
                    # 콜론 형식을 pd.to_datetime이 인식할 수 있게 변환하거나 format 지정
                    dt = pd.to_datetime(dt_str.replace(':', '-', 2)) if ':' in dt_str[:11] else pd.to_datetime(dt_str)
                    
                    self.data_list.append({
                        'pt_path': pt_path, 
                        'actual_dt': dt,
                        'row_data': row.to_dict()
                    })
                except Exception as e:
                    continue
        print(f"✅ 총 {len(self.data_list)}개의 데이터를 추론합니다.")

    def __len__(self): return len(self.data_list)
    def __getitem__(self, idx):
        item = self.data_list[idx]
        data = torch.load(item['pt_path'], map_location='cpu')
        # hidden_states9 로드
        return data[self.layer_key].to(torch.float32), item['actual_dt'], item['row_data']

def collate_fn(batch):
    embeddings, actual_dts, rows = zip(*batch)
    padded = pad_sequence(embeddings, batch_first=True)
    mask = torch.arange(padded.size(1))[None, :] < torch.tensor([e.size(0) for e in embeddings])[:, None]
    return padded, actual_dts, mask, rows

# --- [3. 모델 구조] ---
class SequenceProbeModel(nn.Module):
    def __init__(self, input_dim=4096):
        super().__init__()
        self.attention = nn.Sequential(nn.Linear(input_dim, 512), nn.Tanh(), nn.Linear(512, 1))
        self.head = nn.Sequential(nn.Linear(input_dim, 1024), nn.ReLU(), nn.Dropout(0.2), nn.Linear(1024, 2))
    def forward(self, x, mask):
        attn_scores = self.attention(x).squeeze(-1).masked_fill(~mask, -1e9)
        attn_weights = torch.softmax(attn_scores, dim=1).unsqueeze(-1)
        return self.head(torch.sum(attn_weights * x, dim=1))

# --- [4. 추론 실행 함수] ---
def run_inference():
    # 모델 초기화 및 가중치 로드
    model = SequenceProbeModel().to(DEVICE)
    if not os.path.exists(MODEL_PATH):
        print(f"❌ 가중치 파일을 찾을 수 없습니다: {MODEL_PATH}")
        return
    
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.eval()
    
    # 데이터셋 준비
    dataset = InferenceDataset(PT_DIR, NEW_CSV_PATH)
    loader = DataLoader(dataset, batch_size=1, collate_fn=collate_fn)
    
    results = []
    
    print(f"🚀 추론 시작 (Device: {DEVICE})...")
    with torch.no_grad():
        for h, a_dts, m, rows in tqdm(loader, desc="Inference"):
            h, m = h.to(DEVICE), m.to(DEVICE)
            pred = model(h, m).cpu().numpy()[0]
            
            p_dt = inverse_angular_time(pred[0], pred[1])
            actual_dt = a_dts[0]
            row = rows[0]
            
            diff_sec = calculate_time_diff(actual_dt, p_dt)
            
            # 결과 저장용 딕셔너리 구성
            res_entry = row.copy()
            res_entry['Actual_Time'] = actual_dt.strftime('%Y-%m-%d %H:%M')
            res_entry['Predicted_Time'] = p_dt.strftime('%m-%d %H:%M')
            res_entry['Error_Format'] = str(timedelta(seconds=int(diff_sec)))
            results.append(res_entry)
            
    # CSV 저장
    os.makedirs(os.path.dirname(OUTPUT_SAVE_PATH), exist_ok=True)
    pd.DataFrame(results).to_csv(OUTPUT_SAVE_PATH, index=False, encoding='utf-8-sig')
    print(f"✨ 추론 완료! 결과 저장됨: {OUTPUT_SAVE_PATH}")

if __name__ == "__main__":
    run_inference()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import os

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

def perform_detailed_eda(csv_path):
    if not os.path.exists(csv_path):
        print(f"❌ 파일을 찾을 수 없습니다: {csv_path}")
        return

    # 1. 데이터 로드
    df = pd.read_csv(csv_path)
    
    # Error 문자열을 초 단위 숫자로 변환
    df['Error_Total_Seconds'] = pd.to_timedelta(df['Error_Format']).dt.total_seconds()

    # 날짜 파싱 로직
    def parse_actual(x):
        try:
            return pd.to_datetime(x)
        except:
            return None

    def parse_predicted(x):
        try:
            return datetime.strptime(f"2026-{x}", "%Y-%m-%d %H:%M")
        except:
            return None

    df['Actual_DT'] = df['Actual_Time'].apply(parse_actual)
    df['Pred_DT'] = df['Predicted_Time'].apply(parse_predicted)
    
    # 결측치 제거
    df = df.dropna(subset=['Actual_DT', 'Pred_DT'])
    
    # 분석용 속성 추출
    df['Actual_Month'] = df['Actual_DT'].dt.month
    df['Actual_Hour'] = df['Actual_DT'].dt.hour
    df['Error_Hours'] = df['Error_Total_Seconds'] / 3600
    df['Error_Days'] = df['Error_Hours'] / 24

    print("\n=== 📋 레이어별 예측 오차 요약 통계 ===")
    stats = df['Error_Total_Seconds'].describe()
    print(f"평균 오차: {timedelta(seconds=int(stats['mean']))}")
    print(f"중앙 오차: {timedelta(seconds=int(stats['50%']))}")
    print("="*30)

    # --- 시각화 시작 ---
    fig = plt.figure(figsize=(18, 12))
    
    # (1) 오차 분포
    plt.subplot(2, 2, 1)
    sns.histplot(df['Error_Days'], bins=30, kde=True, color='skyblue')
    plt.title('오차 분포 (단위: 일)')

    # (2) 실제 날짜 vs 예측 날짜 산점도 (윤달 예외 처리 적용)
    plt.subplot(2, 2, 2)
    
    # 🛠️ 수정 포인트: 2월 29일인 경우 2월 28일로 바꿔서 2026년 날짜를 생성합니다.
    def to_2026(dt):
        try:
            return dt.replace(year=2026)
        except ValueError: # 2월 29일 에러 발생 시
            return dt.replace(year=2026, month=2, day=28)

    df['Actual_2026'] = df['Actual_DT'].apply(to_2026)
    df['Actual_DayOfYear'] = df['Actual_2026'].dt.dayofyear
    df['Pred_DayOfYear'] = df['Pred_DT'].dt.dayofyear
    
    plt.scatter(df['Actual_DayOfYear'], df['Pred_DayOfYear'], alpha=0.5, c=df['Error_Days'], cmap='viridis')
    plt.plot([0, 366], [0, 366], 'r--', alpha=0.7)
    plt.title('실제 날짜 vs 예측 날짜 (주기 비교)')
    plt.xlabel('실제 날짜 (Day of Year)')
    plt.ylabel('예측 날짜 (Day of Year)')
    plt.colorbar(label='오차 (Days)')

    # (3) 월별 평균 오차
    plt.subplot(2, 2, 3)
    sns.barplot(data=df, x='Actual_Month', y='Error_Days', palette='coolwarm')
    plt.title('촬영 월별 평균 예측 오차')

    # (4) 시간대별 평균 오차
    plt.subplot(2, 2, 4)
    sns.lineplot(data=df, x='Actual_Hour', y='Error_Hours', marker='o', color='orange')
    plt.xticks(range(0, 24))
    plt.grid(True, alpha=0.3)
    plt.title('촬영 시간대별 평균 예측 오차')

    plt.tight_layout()
    plt.savefig('layer_analysis_plots_fixed.png')
    plt.show()

    # 오차 샘플 출력
    print("\n=== 🚀 가장 잘 맞춘 예측 (Top 5) ===")
    print(df.sort_values('Error_Total_Seconds').head(10)[['File Name', 'Actual_Time', 'Predicted_Time', 'Error_Format']])
    print("\n=== 🚀 가장 못 맞춘 예측 (Top 5) ===")
    print(df.sort_values('Error_Total_Seconds', ascending=False).head(10)[['File Name', 'Actual_Time', 'Predicted_Time', 'Error_Format']])
if __name__ == "__main__":
    path = 'runtime/inference_results_layer9_new.csv'
    perform_detailed_eda(path)

In [ ]:
import os
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
from datetime import datetime, timedelta
import math
import numpy as np
from tqdm import tqdm
import calendar

# --- [0. 설정] ---
RESULT_SAVE_DIR = "analysis_result"
MODEL_SAVE_DIR = "saved_models"
os.makedirs(RESULT_SAVE_DIR, exist_ok=True)
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

# 레이어 설정 (9번 레이어 고정)
LAYER_IDX = 9

# --- [1. 시간 변환 및 오차 계산 함수] ---
DAY_TO_MONTH = 1 / (365 / 12)
MINUTE_TO_HOUR = 1 / 60

def angular_time_representation(T):
    """[month, day, hour, minute] -> [theta, phi] (각도 표현)"""
    month, day, hour, minute = torch.chunk(T.float(), 4, dim=1)
    # 월/일 기반의 연간 주기 (theta)
    month_d = (month - 1) + (day - 1) * DAY_TO_MONTH
    # 시/분 기반의 일간 주기 (phi)
    hour_d = hour + minute * MINUTE_TO_HOUR 
    
    theta = 2 * math.pi * month_d / 12 - math.pi
    phi = 2 * math.pi * hour_d / 24 - math.pi
    return torch.cat((theta, phi), dim=1)

def inverse_angular_time(theta, phi):
    """각도 표현 -> datetime 객체 (2026년 기준)"""
    month_d = ((theta + math.pi) / (2 * math.pi)) * 12
    month = max(1, min(12, int(month_d) + 1))
    
    day_fraction = (month_d - int(month_d)) / DAY_TO_MONTH
    # 2026년 해당 월의 마지막 날 계산
    last_day = calendar.monthrange(2026, month)[1]
    day = max(1, min(last_day, int(day_fraction) + 1))
    
    hour_d = ((phi + math.pi) / (2 * math.pi)) * 24
    hour = int(hour_d) % 24
    minute = int((hour_d - int(hour_d)) * 60) % 60
    
    return datetime(2026, month, day, hour, minute, 0)

def calculate_time_diff(actual_str, pred_dt):
    """실제 문자열 시간과 예측 datetime 간의 초 단위 차이 계산"""
    # 입력 형식 '2012:12:30 23:43:54' 처리
    actual_dt = pd.to_datetime(actual_str, format='%Y:%m:%d %H:%M:%S')
    
    try:
        actual_2026 = actual_dt.replace(year=2026, second=0, microsecond=0)
    except ValueError: # 윤달(2월 29일) 대응
        actual_2026 = actual_dt.replace(year=2026, month=2, day=28, second=0, microsecond=0)
    
    diff = abs((actual_2026 - pred_dt).total_seconds())
    if diff > 15768000: diff = 31536000 - diff # 1년 주기 순환 보정 (반대 방향 거리)
    return diff

def cyclic_loss(pred, target):
    """원형 도메인을 고려한 코사인 손실 함수"""
    return (1 - torch.cos(pred[:, 0] - target[:, 0]) + 1 - torch.cos(pred[:, 1] - target[:, 1])).mean()

# --- [2. 데이터셋 및 모델 정의] ---
class FullSequenceDataset(Dataset):
    def __init__(self, pt_dir, csv_path, layer_idx):
        self.pt_dir = pt_dir
        self.layer_key = f"hidden_states{layer_idx}"
        
        # CSV 로드 및 필터링
        df = pd.read_csv(csv_path)
        
        # 1. Status가 0인 것 제외 (숫자로 변환 후 처리)
        df['Status'] = pd.to_numeric(df['Status'], errors='coerce').fillna(0)
        df = df[df['Status'] != 0].copy()
        
        self.data_list = []
        print(f"📊 데이터셋 준비 중... (총 {len(df)}개 후보)")
        
        for _, row in tqdm(df.iterrows(), total=len(df), desc="데이터 검증"):
            img_name = str(row['File Name']).strip()
            pt_path = os.path.join(pt_dir, f"{img_name}.pt")
            
            if os.path.exists(pt_path):
                try:
                    # 2. DateTime 파싱 ('2012:12:30 23:43:54' 형식 대응)
                    dt = pd.to_datetime(row['DateTime'], format='%Y:%m:%d %H:%M:%S')
                    
                    self.data_list.append({
                        'pt_path': pt_path, 
                        'time_vec': [dt.month, dt.day, dt.hour, dt.minute], 
                        'row_data': row.to_dict()
                    })
                except Exception:
                    continue
        
        print(f"✅ 최종 로드된 데이터: {len(self.data_list)}개")

    def __len__(self): return len(self.data_list)

    def __getitem__(self, idx):
        item = self.data_list[idx]
        # 임베딩 로드
        data = torch.load(item['pt_path'], map_location='cpu')
        embedding = data[self.layer_key].to(torch.float32) # [Sequence, 4096]
        
        return embedding, torch.tensor(item['time_vec'], dtype=torch.float32), item['row_data']

def collate_fn(batch):
    embeddings, times, rows = zip(*batch)
    # 가변 길이 시퀀스 패딩
    padded = pad_sequence(embeddings, batch_first=True)
    # 실제 데이터 위치를 위한 마스크 생성
    mask = torch.arange(padded.size(1))[None, :] < torch.tensor([e.size(0) for e in embeddings])[:, None]
    return padded, torch.stack(times), mask, rows

class SequenceProbeModel(nn.Module):
    def __init__(self, input_dim=4096):
        super().__init__()
        # Attention Layer: 시퀀스 내 중요한 토큰 강조
        self.attention = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.Tanh(),
            nn.Linear(512, 1)
        )
        # Regression Head: 각도 2개(theta, phi) 출력
        self.head = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(1024, 2)
        )

    def forward(self, x, mask):
        # x: [Batch, Seq, Dim], mask: [Batch, Seq]
        attn_scores = self.attention(x).squeeze(-1).masked_fill(~mask, -1e9)
        attn_weights = torch.softmax(attn_scores, dim=1).unsqueeze(-1)
        
        # Context Vector 생성 (가중 평균)
        context = torch.sum(attn_weights * x, dim=1)
        return self.head(context)

# --- [3. 메인 분석 함수] ---
def run_layer_analysis(pt_dir, csv_path):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n{'='*20} 🟢 Layer {LAYER_IDX} 분석 시작 (Device: {device}) {'='*20}")
    
    # 데이터셋 구성
    dataset = FullSequenceDataset(pt_dir, csv_path, LAYER_IDX)
    if len(dataset) < 10:
        print("❌ 학습 데이터가 너무 적습니다. 경로와 Status를 확인하세요.")
        return

    train_len = int(0.8 * len(dataset))
    val_len = len(dataset) - train_len
    train_ds, val_ds = random_split(dataset, [train_len, val_len], generator=torch.Generator().manual_seed(42))
    
    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_ds, batch_size=1, collate_fn=collate_fn)
    
    model = SequenceProbeModel().to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    model_path = os.path.join(MODEL_SAVE_DIR, f"best_probe_layer_{LAYER_IDX}.pth")
    
    best_val_error = float('inf')

    # 학습 루프
    for epoch in range(15):
        # [Train]
        model.train()
        train_loss = 0
        for h, t, m, _ in tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]", leave=False):
            h, t, m = h.to(device), t.to(device), m.to(device)
            
            optimizer.zero_grad()
            target_angular = angular_time_representation(t)
            output = model(h, m)
            
            loss = cyclic_loss(output, target_angular)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # [Validation]
        model.eval()
        total_error_sec = 0
        with torch.no_grad():
            for h, t, m, rows in val_loader:
                h, m = h.to(device), m.to(device)
                pred = model(h, m).cpu().numpy()[0]
                
                p_dt = inverse_angular_time(pred[0], pred[1])
                # 실제 시간과 비교
                total_error_sec += calculate_time_diff(rows[0]['DateTime'], p_dt)

        avg_train_loss = train_loss / len(train_loader)
        avg_val_error_sec = total_error_sec / len(val_loader)
        error_msg = str(timedelta(seconds=int(avg_val_error_sec)))
        
        print(f"Epoch {epoch+1:2d} | Train Loss: {avg_train_loss:.4f} | ⏱️ 평균 오차(검증): {error_msg}")

        if avg_val_error_sec < best_val_error:
            best_val_error = avg_val_error_sec
            torch.save(model.state_dict(), model_path)
            print(f"   ✨ Best Model 저장됨!")

    # --- [4. 최종 결과 저장] ---
    print(f"\n🔍 최적 모델로 최종 리포트 생성 중...")
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    
    final_results = []
    with torch.no_grad():
        for h, t, m, rows in tqdm(val_loader, desc="Final Inference"):
            h, m = h.to(device), m.to(device)
            pred = model(h, m).cpu().numpy()[0]
            row = rows[0]
            
            p_dt = inverse_angular_time(pred[0], pred[1])
            diff_sec = calculate_time_diff(row['DateTime'], p_dt)
            
            res = row.copy()
            res.update({
                'Actual_Parsed': pd.to_datetime(row['DateTime'], format='%Y:%m:%d %H:%M:%S').strftime('%m-%d %H:%M'),
                'Predicted_Time': p_dt.strftime('%m-%d %H:%M'),
                'Error_Duration': str(timedelta(seconds=int(diff_sec)))
            })
            final_results.append(res)

    save_path = os.path.join(RESULT_SAVE_DIR, f"final_report_layer_{LAYER_IDX}.csv")
    pd.DataFrame(final_results).to_csv(save_path, index=False, encoding='utf-8-sig')
    print(f"✨ 분석 완료! 결과 파일: {save_path}")

if __name__ == "__main__":
    # 경로 설정 (사용자 환경에 맞게 수정됨)
    PT_DIR = 'runtime/embedding_results_448_pt_images_only'
    CSV_PATH = 'runtime/results.csv'
    
    run_layer_analysis(PT_DIR, CSV_PATH)

In [ ]:
import os
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
from datetime import datetime, timedelta
import math
import numpy as np
from tqdm import tqdm
import calendar

# --- [0. 설정] ---
RESULT_SAVE_DIR = "analysis_result"
MODEL_SAVE_DIR = "saved_models"
os.makedirs(RESULT_SAVE_DIR, exist_ok=True)
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

# 레이어 설정 (9번 레이어 고정)
LAYER_IDX = 9

# --- [1. 시간 변환 및 오차 계산 함수] ---
DAY_TO_MONTH = 1 / (365 / 12)
MINUTE_TO_HOUR = 1 / 60

def angular_time_representation(T):
    """[month, day, hour, minute] -> [theta, phi] (각도 표현)"""
    month, day, hour, minute = torch.chunk(T.float(), 4, dim=1)
    # 월/일 기반의 연간 주기 (theta)
    month_d = (month - 1) + (day - 1) * DAY_TO_MONTH
    # 시/분 기반의 일간 주기 (phi)
    hour_d = hour + minute * MINUTE_TO_HOUR 
    
    theta = 2 * math.pi * month_d / 12 - math.pi
    phi = 2 * math.pi * hour_d / 24 - math.pi
    return torch.cat((theta, phi), dim=1)

def inverse_angular_time(theta, phi):
    """각도 표현 -> datetime 객체 (2026년 기준)"""
    month_d = ((theta + math.pi) / (2 * math.pi)) * 12
    month = max(1, min(12, int(month_d) + 1))
    
    day_fraction = (month_d - int(month_d)) / DAY_TO_MONTH
    # 2026년 해당 월의 마지막 날 계산
    last_day = calendar.monthrange(2026, month)[1]
    day = max(1, min(last_day, int(day_fraction) + 1))
    
    hour_d = ((phi + math.pi) / (2 * math.pi)) * 24
    hour = int(hour_d) % 24
    minute = int((hour_d - int(hour_d)) * 60) % 60
    
    return datetime(2026, month, day, hour, minute, 0)

def calculate_time_diff(actual_str, pred_dt):
    """실제 문자열 시간과 예측 datetime 간의 초 단위 차이 계산"""
    # 입력 형식 '2012:12:30 23:43:54' 처리
    actual_dt = pd.to_datetime(actual_str, format='%Y:%m:%d %H:%M:%S')
    
    try:
        actual_2026 = actual_dt.replace(year=2026, second=0, microsecond=0)
    except ValueError: # 윤달(2월 29일) 대응
        actual_2026 = actual_dt.replace(year=2026, month=2, day=28, second=0, microsecond=0)
    
    diff = abs((actual_2026 - pred_dt).total_seconds())
    if diff > 15768000: diff = 31536000 - diff # 1년 주기 순환 보정 (반대 방향 거리)
    return diff

def cyclic_loss(pred, target):
    """원형 도메인을 고려한 코사인 손실 함수"""
    return (1 - torch.cos(pred[:, 0] - target[:, 0]) + 1 - torch.cos(pred[:, 1] - target[:, 1])).mean()

# --- [2. 데이터셋 및 모델 정의] ---
class FullSequenceDataset(Dataset):
    def __init__(self, pt_dir, csv_path, layer_idx):
        self.pt_dir = pt_dir
        self.layer_key = f"hidden_states{layer_idx}"
        
        # CSV 로드 및 필터링
        df = pd.read_csv(csv_path)
        
        # 1. Status가 0인 것 제외 (숫자로 변환 후 처리)
        df['Status'] = pd.to_numeric(df['Status'], errors='coerce').fillna(0)
        df = df[df['Status'] != 0].copy()
        
        self.data_list = []
        print(f"📊 데이터셋 준비 중... (총 {len(df)}개 후보)")
        
        for _, row in tqdm(df.iterrows(), total=len(df), desc="데이터 검증"):
            img_name = str(row['File Name']).strip()
            pt_path = os.path.join(pt_dir, f"{img_name}.pt")
            
            if os.path.exists(pt_path):
                try:
                    # 2. DateTime 파싱 ('2012:12:30 23:43:54' 형식 대응)
                    dt = pd.to_datetime(row['DateTime'], format='%Y:%m:%d %H:%M:%S')
                    
                    self.data_list.append({
                        'pt_path': pt_path, 
                        'time_vec': [dt.month, dt.day, dt.hour, dt.minute], 
                        'row_data': row.to_dict()
                    })
                except Exception:
                    continue
        
        print(f"✅ 최종 로드된 데이터: {len(self.data_list)}개")

    def __len__(self): return len(self.data_list)

    def __getitem__(self, idx):
        item = self.data_list[idx]
        # 임베딩 로드
        data = torch.load(item['pt_path'], map_location='cpu')
        embedding = data[self.layer_key].to(torch.float32) # [Sequence, 4096]
        
        return embedding, torch.tensor(item['time_vec'], dtype=torch.float32), item['row_data']

def collate_fn(batch):
    embeddings, times, rows = zip(*batch)
    # 가변 길이 시퀀스 패딩
    padded = pad_sequence(embeddings, batch_first=True)
    # 실제 데이터 위치를 위한 마스크 생성
    mask = torch.arange(padded.size(1))[None, :] < torch.tensor([e.size(0) for e in embeddings])[:, None]
    return padded, torch.stack(times), mask, rows

class SequenceProbeModel(nn.Module):
    def __init__(self, input_dim=4096):
        super().__init__()
        # Attention Layer: 시퀀스 내 중요한 토큰 강조
        self.attention = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.Tanh(),
            nn.Linear(512, 1)
        )
        # Regression Head: 각도 2개(theta, phi) 출력
        self.head = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(1024, 2)
        )

    def forward(self, x, mask):
        # x: [Batch, Seq, Dim], mask: [Batch, Seq]
        attn_scores = self.attention(x).squeeze(-1).masked_fill(~mask, -1e9)
        attn_weights = torch.softmax(attn_scores, dim=1).unsqueeze(-1)
        
        # Context Vector 생성 (가중 평균)
        context = torch.sum(attn_weights * x, dim=1)
        return self.head(context)

# --- [3. 메인 분석 함수] ---
def run_layer_analysis(pt_dir, csv_path):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n{'='*20} 🟢 Layer {LAYER_IDX} 분석 시작 (Device: {device}) {'='*20}")
    
    # 데이터셋 구성
    dataset = FullSequenceDataset(pt_dir, csv_path, LAYER_IDX)
    if len(dataset) < 10:
        print("❌ 학습 데이터가 너무 적습니다. 경로와 Status를 확인하세요.")
        return

    train_len = int(0.8 * len(dataset))
    val_len = len(dataset) - train_len
    train_ds, val_ds = random_split(dataset, [train_len, val_len], generator=torch.Generator().manual_seed(42))
    
    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_ds, batch_size=1, collate_fn=collate_fn)
    
    model = SequenceProbeModel().to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-4)
    model_path = os.path.join(MODEL_SAVE_DIR, f"best_probe_layer_{LAYER_IDX}.pth")
    
    best_val_error = float('inf')

    # 학습 루프
    for epoch in range(15):
        # [Train]
        model.train()
        train_loss = 0
        for h, t, m, _ in tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]", leave=False):
            h, t, m = h.to(device), t.to(device), m.to(device)
            
            optimizer.zero_grad()
            target_angular = angular_time_representation(t)
            output = model(h, m)
            
            loss = cyclic_loss(output, target_angular)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # [Validation]
        model.eval()
        total_error_sec = 0
        with torch.no_grad():
            for h, t, m, rows in val_loader:
                h, m = h.to(device), m.to(device)
                pred = model(h, m).cpu().numpy()[0]
                
                p_dt = inverse_angular_time(pred[0], pred[1])
                # 실제 시간과 비교
                total_error_sec += calculate_time_diff(rows[0]['DateTime'], p_dt)

        avg_train_loss = train_loss / len(train_loader)
        avg_val_error_sec = total_error_sec / len(val_loader)
        error_msg = str(timedelta(seconds=int(avg_val_error_sec)))
        
        print(f"Epoch {epoch+1:2d} | Train Loss: {avg_train_loss:.4f} | ⏱️ 평균 오차(검증): {error_msg}")

        if avg_val_error_sec < best_val_error:
            best_val_error = avg_val_error_sec
            torch.save(model.state_dict(), model_path)
            print(f"   ✨ Best Model 저장됨!")

    # --- [4. 최종 결과 저장] ---
    print(f"\n🔍 최적 모델로 최종 리포트 생성 중...")
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    
    final_results = []
    with torch.no_grad():
        for h, t, m, rows in tqdm(val_loader, desc="Final Inference"):
            h, m = h.to(device), m.to(device)
            pred = model(h, m).cpu().numpy()[0]
            row = rows[0]
            
            p_dt = inverse_angular_time(pred[0], pred[1])
            diff_sec = calculate_time_diff(row['DateTime'], p_dt)
            
            res = row.copy()
            res.update({
                'Actual_Parsed': pd.to_datetime(row['DateTime'], format='%Y:%m:%d %H:%M:%S').strftime('%m-%d %H:%M'),
                'Predicted_Time': p_dt.strftime('%m-%d %H:%M'),
                'Error_Duration': str(timedelta(seconds=int(diff_sec)))
            })
            final_results.append(res)

    save_path = os.path.join(RESULT_SAVE_DIR, f"final_report_layer_every_{LAYER_IDX}.csv")
    pd.DataFrame(final_results).to_csv(save_path, index=False, encoding='utf-8-sig')
    print(f"✨ 분석 완료! 결과 파일: {save_path}")

if __name__ == "__main__":
    # 경로 설정 (사용자 환경에 맞게 수정됨)
    PT_DIR = 'runtime/embedding_results_448_pt'
    CSV_PATH = 'runtime/results.csv'
    
    run_layer_analysis(PT_DIR, CSV_PATH)

In [ ]:
import os
import math
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torch.nn.utils.rnn import pad_sequence
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from tqdm import tqdm
import calendar

# --- [1. 유틸리티 및 시간 변환 함수] ---

def safe_replace_year(dt, target_year=2026):
    """2월 29일 데이터를 평년(2026)으로 바꿀 때 발생하는 ValueError 방지"""
    try:
        return dt.replace(year=target_year, second=0, microsecond=0)
    except ValueError:
        # 2월 29일인 경우 2월 28일로 조정
        return dt.replace(year=target_year, month=2, day=28, second=0, microsecond=0)

def angular_time_representation(T):
    month, day, hour, minute, second = torch.chunk(T.float(), 5, dim=1)
    DAY_TO_MONTH = 1 / (365/12)
    month_d = (month - 1) + (day - 1) * DAY_TO_MONTH
    hour_d = hour + minute * (1/60) + second * (1/3600)
    
    theta = 2 * math.pi * month_d / 12 - math.pi
    phi = 2 * math.pi * hour_d / 24 - math.pi
    return torch.cat((theta, phi), dim=1)

def inverse_angular_time(theta, phi):
    month_d = ((theta + math.pi) / (2 * math.pi)) * 12
    month = max(1, min(12, int(month_d) + 1))
    day_fraction = (month_d - int(month_d)) * (365 / 12)
    last_day = calendar.monthrange(2026, month)[1]
    day = max(1, min(last_day, int(day_fraction) + 1))
    hour_d = ((phi + math.pi) / (2 * math.pi)) * 24
    hour = int(hour_d) % 24
    minute = int((hour_d - int(hour_d)) * 60) % 60
    return datetime(2026, month, day, hour, minute, 0)

# --- [2. 모델: Robust Residual Frequency-Aware Probe] ---

class RobustFrequencyProbe(nn.Module):
    def __init__(self, input_dim=4096, d_model=512, num_queries=10):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        
        sigma_list = [2**i for i in range(num_queries)]
        init_queries = torch.randn(num_queries, d_model)
        for i, sigma in enumerate(sigma_list):
            init_queries[i] *= (1.0 / (sigma + 1e-6))
        self.query_bank = nn.Parameter(init_queries)
        
        self.cross_attn = nn.MultiheadAttention(d_model, num_heads=8, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, 1024),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(1024, d_model)
        )
        self.norm2 = nn.LayerNorm(d_model)
        
        self.head = nn.Sequential(
            nn.Linear(num_queries * d_model, 1024),
            nn.LayerNorm(1024),
            nn.GELU(),
            nn.Linear(1024, 4) 
        )

    def forward(self, x, mask):
        batch_size = x.size(0)
        img_feats = self.input_proj(x)
        queries = self.query_bank.unsqueeze(0).expand(batch_size, -1, -1)
        
        attn_out, _ = self.cross_attn(query=queries, key=img_feats, value=img_feats, key_padding_mask=~mask)
        x_queries = self.norm1(queries + attn_out)
        x_queries = self.norm2(x_queries + self.ffn(x_queries))
        
        out = self.head(x_queries.reshape(batch_size, -1))
        
        mu = out[:, :2]
        log_var = out[:, 2:]
        clamped_log_var = torch.clamp(log_var, min=-5.0, max=3.0)
        return torch.cat([mu, clamped_log_var], dim=1)

# --- [3. 손실 함수] ---

def balanced_geo_loss(pred, target, lat_weights, reg_weight=0.05):
    theta, phi, log_var_s, log_var_t = pred[:, 0], pred[:, 1], pred[:, 2], pred[:, 3]
    loss_s = 1 - torch.cos(theta - target[:, 0])
    loss_t = 1 - torch.cos(phi - target[:, 1])
    
    weighted_loss_s = (lat_weights * loss_s) * torch.exp(-log_var_s) + log_var_s
    weighted_loss_t = loss_t * torch.exp(-log_var_t) + log_var_t
    reg_loss = reg_weight * (torch.abs(log_var_s) + torch.abs(log_var_t)).mean()
    
    return (weighted_loss_s + weighted_loss_t).mean() + reg_loss, loss_s.mean(), loss_t.mean()

# --- [4. 데이터셋 및 실행부] ---

class GlobalTimeDataset(Dataset):
    def __init__(self, pt_dir, csv_path, layer_idx=9):
        self.pt_dir = pt_dir
        self.layer_key = f"hidden_states{layer_idx}"
        df = pd.read_csv(csv_path)
        self.data_list = []
        
        print(f"DEBUG: 데이터 필터링 중...")
        for _, row in df.iterrows():
            img_name = str(row['File Name']).strip()
            pt_path = os.path.join(pt_dir, f"{img_name}.pt")
            if not os.path.exists(pt_path): continue
            
            try:
                dt = pd.to_datetime(str(row['DateTime']).strip())
                lat = float(row['latitude']) if pd.notnull(row['latitude']) else 37.0
                is_south = lat < -5.0
                if is_south: dt = dt + pd.DateOffset(months=6)
                
                lat_weight = min(1.0, abs(lat) / 37.0 + 0.1)
                self.data_list.append({
                    'pt_path': pt_path, 
                    'time_vec': [dt.month, dt.day, dt.hour, dt.minute, dt.second],
                    'lat_weight': lat_weight, 'is_south': is_south, 'row_data': row.to_dict()
                })
            except: continue
        print(f"DEBUG: 총 유효 샘플: {len(self.data_list)}")

    def __len__(self): return len(self.data_list)
    def __getitem__(self, idx):
        item = self.data_list[idx]
        data = torch.load(item['pt_path'], map_location='cpu')
        return (data[self.layer_key].to(torch.float32), 
                torch.tensor(item['time_vec'], dtype=torch.float32),
                torch.tensor(item['lat_weight'], dtype=torch.float32), 
                item['is_south'], item['row_data'])

def collate_fn(batch):
    embeddings, times, weights, south_flags, rows = zip(*batch)
    padded = pad_sequence(embeddings, batch_first=True)
    mask = torch.arange(padded.size(1))[None, :] < torch.tensor([e.size(0) for e in embeddings])[:, None]
    return padded, torch.stack(times), mask, torch.stack(weights), south_flags, rows

# --- [5. 메인 트레이닝 루프] ---

def run_training(pt_dir, csv_path):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"\n🚀 트레이닝 시작 (Device: {device})")

    dataset = GlobalTimeDataset(pt_dir, csv_path)
    train_len = int(0.8 * len(dataset))
    train_ds, val_ds = random_split(dataset, [train_len, len(dataset)-train_len], generator=torch.Generator().manual_seed(42))
    
    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True, collate_fn=collate_fn)
    val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, collate_fn=collate_fn) # 배치 사이즈 증가

    model = RobustFrequencyProbe().to(device)
    optimizer = optim.AdamW(model.parameters(), lr=8e-5, weight_decay=0.02)
    
    best_val_loss = float('inf')
    
    for epoch in range(25):
        # [Train]
        model.train()
        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1:02d} [Train]")
        sum_train_loss = 0
        for h, t, m, w, _, _ in train_pbar:
            h, t, m, w = h.to(device), t.to(device), m.to(device), w.to(device)
            optimizer.zero_grad()
            pred = model(h, m)
            loss, _, _ = balanced_geo_loss(pred, angular_time_representation(t), w)
            loss.backward()
            optimizer.step()
            sum_train_loss += loss.item()

        # [Validation] - 전체 val_loss 및 MAE 계산
        model.eval()
        sum_val_loss = 0
        total_error_sec = 0
        val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1:02d} [Val]", leave=False)
        
        with torch.no_grad():
            for h, t, m, w, south_flags, rows in val_pbar:
                h, t, m, w = h.to(device), t.to(device), m.to(device), w.to(device)
                pred = model(h, m)
                v_loss, _, _ = balanced_geo_loss(pred, angular_time_representation(t), w)
                sum_val_loss += v_loss.item()
                
                # 오차(MAE) 계산
                pred_np = pred.cpu().numpy()
                for i in range(len(pred_np)):
                    p_dt = inverse_angular_time(pred_np[i, 0], pred_np[i, 1])
                    if south_flags[i]: p_dt = p_dt - pd.DateOffset(months=6)
                    
                    actual_raw = pd.to_datetime(rows[i]['DateTime'])
                    actual_dt = safe_replace_year(actual_raw, 2026)
                    
                    diff = abs((actual_dt - p_dt).total_seconds())
                    if diff > 15768000: diff = 31536000 - diff
                    total_error_sec += diff

        avg_train_loss = sum_train_loss / len(train_loader)
        avg_val_loss = sum_val_loss / len(val_loader)
        avg_mae = total_error_sec / len(val_ds)
        
        print(f"\n✅ Epoch {epoch+1:02d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
        print(f"⏱️ MAE: {str(timedelta(seconds=int(avg_mae)))}")
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), f"best_model_val_loss.pth")
            print(f"✨ Best Model Saved (Val Loss 기준) ✨")
        print("-" * 50)

if __name__ == "__main__":
    PT_DIR = 'runtime/embedding_results_448_pt_coco'
    CSV_PATH = 'runtime/final_data.csv'
    run_training(PT_DIR, CSV_PATH)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import os

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

def perform_detailed_eda(csv_path):
    if not os.path.exists(csv_path):
        print(f"❌ 파일을 찾을 수 없습니다: {csv_path}")
        return

    # 1. 데이터 로드
    df = pd.read_csv(csv_path)
    
    # Error 문자열을 초 단위 숫자로 변환
    df['Error_Total_Seconds'] = pd.to_timedelta(df['Error_Format']).dt.total_seconds()

    # 날짜 파싱 로직
    def parse_actual(x):
        try:
            return pd.to_datetime(x)
        except:
            return None

    def parse_predicted(x):
        try:
            return datetime.strptime(f"2026-{x}", "%Y-%m-%d %H:%M")
        except:
            return None

    df['Actual_DT'] = df['Actual_Time'].apply(parse_actual)
    df['Pred_DT'] = df['Predicted_Time'].apply(parse_predicted)
    
    # 결측치 제거
    df = df.dropna(subset=['Actual_DT', 'Pred_DT'])
    
    # 분석용 속성 추출
    df['Actual_Month'] = df['Actual_DT'].dt.month
    df['Actual_Hour'] = df['Actual_DT'].dt.hour
    df['Error_Hours'] = df['Error_Total_Seconds'] / 3600
    df['Error_Days'] = df['Error_Hours'] / 24

    print("\n=== 📋 레이어별 예측 오차 요약 통계 ===")
    stats = df['Error_Total_Seconds'].describe()
    print(f"평균 오차: {timedelta(seconds=int(stats['mean']))}")
    print(f"중앙 오차: {timedelta(seconds=int(stats['50%']))}")
    print("="*30)

    # --- 시각화 시작 ---
    fig = plt.figure(figsize=(18, 12))
    
    # (1) 오차 분포
    plt.subplot(2, 2, 1)
    sns.histplot(df['Error_Days'], bins=30, kde=True, color='skyblue')
    plt.title('오차 분포 (단위: 일)')

    # (2) 실제 날짜 vs 예측 날짜 산점도 (윤달 예외 처리 적용)
    plt.subplot(2, 2, 2)
    
    # 🛠️ 수정 포인트: 2월 29일인 경우 2월 28일로 바꿔서 2026년 날짜를 생성합니다.
    def to_2026(dt):
        try:
            return dt.replace(year=2026)
        except ValueError: # 2월 29일 에러 발생 시
            return dt.replace(year=2026, month=2, day=28)

    df['Actual_2026'] = df['Actual_DT'].apply(to_2026)
    df['Actual_DayOfYear'] = df['Actual_2026'].dt.dayofyear
    df['Pred_DayOfYear'] = df['Pred_DT'].dt.dayofyear
    
    plt.scatter(df['Actual_DayOfYear'], df['Pred_DayOfYear'], alpha=0.5, c=df['Error_Days'], cmap='viridis')
    plt.plot([0, 366], [0, 366], 'r--', alpha=0.7)
    plt.title('실제 날짜 vs 예측 날짜 (주기 비교)')
    plt.xlabel('실제 날짜 (Day of Year)')
    plt.ylabel('예측 날짜 (Day of Year)')
    plt.colorbar(label='오차 (Days)')

    # (3) 월별 평균 오차
    plt.subplot(2, 2, 3)
    sns.barplot(data=df, x='Actual_Month', y='Error_Days', palette='coolwarm')
    plt.title('촬영 월별 평균 예측 오차')

    # (4) 시간대별 평균 오차
    plt.subplot(2, 2, 4)
    sns.lineplot(data=df, x='Actual_Hour', y='Error_Hours', marker='o', color='orange')
    plt.xticks(range(0, 24))
    plt.grid(True, alpha=0.3)
    plt.title('촬영 시간대별 평균 예측 오차')

    plt.tight_layout()
    plt.savefig('layer_analysis_plots_fixed.png')
    plt.show()

    # 오차 샘플 출력
    print("\n=== 🚀 가장 잘 맞춘 예측 (Top 5) ===")
    print(df.sort_values('Error_Total_Seconds').head(10)[['File Name', 'Actual_Time', 'Predicted_Time', 'Error_Format']])
    print("\n=== 🚀 가장 못 맞춘 예측 (Top 5) ===")
    print(df.sort_values('Error_Total_Seconds', ascending=False).head(10)[['File Name', 'Actual_Time', 'Predicted_Time', 'Error_Format']])
if __name__ == "__main__":
    path = 'runtime/inference_results_final.csv'
    perform_detailed_eda(path)